# 🔍 Consultas a la Base de Datos
Este notebook recorre los patrones de consulta más comunes con SQLAlchemy ORM.

**Índice de secciones:**
1. Setup e imports
2. Consulta simple — `select`
3. Ordenar resultados — `order_by`
4. Filtrar con `where`
5. Múltiples condiciones — `and_` / `or_`
6. Un único resultado — `scalar` / `one` / `one_or_none`
7. Relaciones — acceso por atributo (lazy loading)
8. Carga anticipada — `joinedload` / `selectinload`
9. Funciones de agregado — `count`, `avg`, `max`, `min`
10. Joins — `join` / `outerjoin`
11. Agrupación — `group_by` + `having`
12. Paginación — `limit` + `offset`
13. `scalars()` vs `execute()` vs `mappings()`
14. Modificar y eliminar registros

---
## 1️⃣ Setup e imports

In [2]:
import sys
sys.path.append("../src")

In [3]:
from datetime import date
from sqlalchemy import select, func, desc, asc, and_, or_
from sqlalchemy.orm import joinedload, selectinload

from domain import (
    engine, Session,
    Patient, Staff, SampleType, Container, 
    ResearchProject, Sample, Protocol,
    LogTemperature, QualityControl,
    ResearchProjectSamples, ProjectTeam,
)



✅ Configuración cargada: development -> sqlite:///biotrack.db


ModuleNotFoundError: No module named 'protocol'

---
## 2️⃣ Consulta simple — `select`
Recupera todos los registros de una tabla.
- `session.scalars(stmt).all()` → devuelve una lista de objetos ORM

In [1]:
# Todos los pacientes
with Session() as session:
    stmt = select(Patient)
    patients = session.scalars(stmt).all()

    for p in patients:
        print(p.id, p.code, p.name, p.lastname, p.birth_date, p.active)

NameError: name 'Session' is not defined

In [ ]:
# Todas las muestras
with Session() as session:
    stmt = select(Sample)
    samples = session.scalars(stmt).all()

    for s in samples:
        print(s.id, s.code, s.volume, s.status, s.extraction_date)

---
## 3️⃣ Ordenar resultados — `order_by`
- `asc(...)` → ascendente (por defecto)
- `desc(...)` → descendente

In [ ]:
# Pacientes ordenados por apellido (A → Z)
with Session() as session:
    stmt = select(Patient).order_by(Patient.lastname)
    patients = session.scalars(stmt).all()

    for p in patients:
        print(p.lastname, p.name)

In [ ]:
# Muestras ordenadas por fecha de extracción, las más recientes primero
with Session() as session:
    stmt = select(Sample).order_by(desc(Sample.extraction_date))
    samples = session.scalars(stmt).all()

    for s in samples:
        print(s.code, s.extraction_date, s.status)

---
## 4️⃣ Filtrar con `where`

In [ ]:
# Solo pacientes activos
with Session() as session:
    stmt = select(Patient).where(Patient.active == True)
    patients = session.scalars(stmt).all()

    for p in patients:
        print(p.code, p.name, p.lastname)

In [ ]:
# Muestras con estado 'pending'
with Session() as session:
    stmt = select(Sample).where(Sample.status == "pending")
    samples = session.scalars(stmt).all()

    for s in samples:
        print(s.code, s.status)

In [ ]:
# Muestras extraídas después del 2024-01-01
with Session() as session:
    stmt = select(Sample).where(Sample.extraction_date >= date(2024, 1, 1))
    samples = session.scalars(stmt).all()

    for s in samples:
        print(s.code, s.extraction_date)

In [ ]:
# Staff cuyo nombre contiene 'Laura'
with Session() as session:
    stmt = select(Staff).where(Staff.name.like("%Laura%"))
    staff_list = session.scalars(stmt).all()

    for s in staff_list:
        print(s.code, s.name, s.lastname, s.role)

---
## 5️⃣ Múltiples condiciones — `and_` / `or_`
- `and_(cond1, cond2)` → las dos deben cumplirse
- `or_(cond1, cond2)` → basta con que una se cumpla

In [ ]:
# Staff activo Y con rol 'researcher'
with Session() as session:
    stmt = (
        select(Staff)
        .where(
            and_(
                Staff.active == True,
                Staff.role == "researcher"
            )
        )
        .order_by(Staff.lastname)
    )

    staff_list = session.scalars(stmt).all()

    for s in staff_list:
        print(s.code, s.name, s.lastname, s.role, s.active)

In [ ]:
# Muestras con volumen > 5 Y estado 'in_process' o 'analyzed'
with Session() as session:
    stmt = (
        select(Sample)
        .where(
            and_(
                Sample.volume > 5,
                or_(
                    Sample.status == "in_process",
                    Sample.status == "analyzed"
                )
            )
        )
        .order_by(Sample.volume)
    )

    samples = session.scalars(stmt).all()

    for s in samples:
        print(s.code, float(s.volume), s.status)

In [ ]:
# Staff con rol 'manager' O 'administrator'
with Session() as session:
    stmt = (
        select(Staff)
        .where(
            or_(
                Staff.role == "manager",
                Staff.role == "administrator"
            )
        )
    )

    staff_list = session.scalars(stmt).all()

    for s in staff_list:
        print(s.code, s.name, s.role)

---
## 6️⃣ Un único resultado
| Método | Devuelve | Si no hay | Si hay más de uno |
|---|---|---|---|
| `scalar(stmt)` | objeto o `None` | `None` | devuelve el primero |
| `scalars(stmt).one()` | objeto | lanza error | lanza error |
| `scalars(stmt).one_or_none()` | objeto o `None` | `None` | lanza error |

In [ ]:
# scalar() — útil cuando buscas por campo único (code, email...)
# Devuelve None si no existe, no lanza excepción
with Session() as session:
    stmt = select(Patient).where(Patient.code == "P001")
    patient = session.scalar(stmt)

    if patient:
        print(patient.code, patient.name, patient.lastname)

In [ ]:
# one() — cuando SABES con certeza que debe existir exactamente uno
# Lanza NoResultFound si no hay, MultipleResultsFound si hay más de uno
with Session() as session:
    stmt = select(Sample).where(Sample.code == "M001")
    sample = session.scalars(stmt).one()
    print(sample.code, sample.status)

In [ ]:
# one_or_none() — cuando puede existir o no, pero nunca duplicados
# Devuelve None si no hay, lanza error si hay más de uno
with Session() as session:
    stmt = select(Protocol).where(Protocol.code == "PROT-999")
    protocol = session.scalars(stmt).one_or_none()

    if protocol:
        print(protocol.code, protocol.name)
    else:
        print("Protocolo no encontrado")

---
## 7️⃣ Relaciones — acceso por atributo (lazy loading)
SQLAlchemy puede cargar los objetos relacionados automáticamente cuando accedes al atributo.

⚠️ Esto funciona **dentro de la sesión**. Si la sesión ya está cerrada, lanza `DetachedInstanceError`.
Para usar los datos fuera de la sesión → ver sección 8 (eager loading).

In [ ]:
# Paciente → sus muestras (1:N)
with Session() as session:
    patient = session.scalar(select(Patient).where(Patient.code == "P001"))

    if patient:
        print(f"Paciente: {patient.name} {patient.lastname}")
        print(f"Número de muestras: {len(patient.samples)}")
        for sample in patient.samples:
            print(f"  → {sample.code} | {sample.status} | {sample.extraction_date}")

In [ ]:
# Muestra → paciente, tipo, contenedor, control de calidad (N:1 y 1:1)
with Session() as session:
    sample = session.scalar(select(Sample).where(Sample.code == "M001"))

    if sample:
        print(f"Muestra:    {sample.code}")
        print(f"Paciente:   {sample.patient.name} {sample.patient.lastname}")
        print(f"Tipo:       {sample.sample_type.type_name}")
        print(f"Contenedor: {sample.container.code}")
        if sample.quality_control:
            print(f"QC result:  {sample.quality_control.result}")

In [ ]:
# Muestra → sus protocolos (N:M)
with Session() as session:
    sample = session.scalar(select(Sample).where(Sample.code == "M001"))

    if sample:
        print(f"Muestra: {sample.code}")
        for protocol in sample.protocols:
            print(f"  → {protocol.code} | {protocol.name}")

In [ ]:
# Staff → sus proyectos de investigación (N:M a través de project_team)
with Session() as session:
    staff = session.scalar(select(Staff).where(Staff.code == "S001"))

    if staff:
        print(f"Trabajador: {staff.name} {staff.lastname} ({staff.role})")
        for project in staff.research_projects:
            print(f"  → {project.project_name} | inicio: {project.start_date}")

---
## 8️⃣ Carga anticipada — `joinedload` / `selectinload`
Carga los objetos relacionados en la misma consulta, antes de cerrar la sesión.
Imprescindible si quieres usar los datos **fuera** del bloque `with`.

| Opción | Cuándo usarla |
|---|---|
| `joinedload` | relaciones N:1 o 1:1 (un solo objeto relacionado) |
| `selectinload` | relaciones 1:N o N:M (listas de objetos relacionados) |

In [ ]:
# joinedload → muestra + su paciente en una sola query SQL (JOIN)
# Ideal para N:1: cada muestra tiene UN paciente
with Session() as session:
    stmt = (
        select(Sample)
        .options(joinedload(Sample.patient))
        .options(joinedload(Sample.sample_type))
        .order_by(Sample.extraction_date)
    )
    samples = session.scalars(stmt).all()

    for s in samples:
        print(f"{s.code} | {s.patient.name} {s.patient.lastname} | {s.sample_type.type_name}")

In [ ]:
# selectinload → paciente + todas sus muestras (1:N)
# Hace una segunda query: SELECT * FROM sample WHERE id_patient IN (...)
with Session() as session:
    stmt = (
        select(Patient)
        .options(selectinload(Patient.samples))
        .where(Patient.active == True)
    )
    patients = session.scalars(stmt).all()

    for p in patients:
        print(f"{p.name} {p.lastname} — {len(p.samples)} muestra(s)")
        for s in p.samples:
            print(f"   {s.code} | {s.status}")

In [ ]:
# selectinload → muestra + sus protocolos (N:M)
with Session() as session:
    stmt = (
        select(Sample)
        .options(selectinload(Sample.protocols))
        .options(joinedload(Sample.patient))
    )
    samples = session.scalars(stmt).all()

    for s in samples:
        nombres_protocolos = [p.name for p in s.protocols]
        print(f"{s.code} ({s.patient.name}) → protocolos: {nombres_protocolos or 'ninguno'}")

---
## 9️⃣ Funciones de agregado — `count`, `avg`, `max`, `min`
Cuando seleccionas columnas sueltas (no objetos ORM enteros), usa `session.execute()` en lugar de `scalars()`.

In [ ]:
# Total de muestras en la BD
with Session() as session:
    stmt = select(func.count(Sample.id))
    total = session.scalar(stmt)
    print(f"Total de muestras: {total}")

In [ ]:
# Volumen medio, máximo y mínimo de todas las muestras
with Session() as session:
    stmt = select(
        func.avg(Sample.volume).label("promedio"),
        func.max(Sample.volume).label("maximo"),
        func.min(Sample.volume).label("minimo")
    )

    row = session.execute(stmt).one()
    print(f"Promedio: {float(row.promedio):.4f}")
    print(f"Máximo:   {float(row.maximo):.4f}")
    print(f"Mínimo:   {float(row.minimo):.4f}")

In [ ]:
# Muestras por estado
with Session() as session:
    stmt = (
        select(Sample.status, func.count(Sample.id).label("total"))
        .group_by(Sample.status)
        .order_by(desc("total"))
    )

    rows = session.execute(stmt).all()

    for status, total in rows:
        print(f"{status:<15} {total}")

---
## 🔟 Joins — `join` / `outerjoin`
- `join` → INNER JOIN: solo filas que tienen relación en ambas tablas
- `outerjoin` → LEFT JOIN: todas las filas de la izquierda, aunque no tengan relación

In [ ]:
# INNER JOIN: muestra + paciente + tipo (solo muestras que tienen todo)
with Session() as session:
    stmt = (
        select(Sample.code, Patient.name, Patient.lastname, SampleType.type_name)
        .join(Patient, Sample.id_patient == Patient.id)
        .join(SampleType, Sample.id_sample_type == SampleType.id)
        .order_by(Sample.code)
    )

    rows = session.execute(stmt).all()

    for sample_code, name, lastname, type_name in rows:
        print(f"{sample_code} | {name} {lastname} | {type_name}")

In [ ]:
# Alternativa con la relación directamente (más legible)
with Session() as session:
    stmt = (
        select(Sample.code, Patient.name, SampleType.type_name)
        .join(Sample.patient)
        .join(Sample.sample_type)
        .order_by(Patient.lastname)
    )

    rows = session.execute(stmt).all()

    for code, name, type_name in rows:
        print(f"{code} | {name} | {type_name}")

In [ ]:
# LEFT JOIN: todos los pacientes, tengan muestras o no
# func.count() cuenta las muestras (0 si no tiene ninguna)
with Session() as session:
    stmt = (
        select(Patient.name, Patient.lastname, func.count(Sample.id).label("num_muestras"))
        .outerjoin(Sample, Sample.id_patient == Patient.id)
        .group_by(Patient.id, Patient.name, Patient.lastname)
        .order_by(desc("num_muestras"), Patient.lastname)
    )

    rows = session.execute(stmt).all()

    for name, lastname, num in rows:
        print(f"{name} {lastname:<20} → {num} muestra(s)")

In [ ]:
# LEFT JOIN: todos los protocolos y quién los revisó (puede ser NULL)
with Session() as session:
    stmt = (
        select(
            Protocol.code,
            Protocol.name,
            Staff.name.label("revisor")
        )
        .outerjoin(Staff, Protocol.reviewed_by_id == Staff.id)
        .order_by(Protocol.code)
    )

    rows = session.execute(stmt).all()

    for code, name, revisor in rows:
        print(f"{code} | {name:<40} | revisor: {revisor or 'sin asignar'}")

---
## 1️⃣1️⃣ Agrupación — `group_by` + `having`
`having` es el `where` que se aplica **después** de agrupar (sobre resultados de agregado).

In [ ]:
# Número de muestras por tipo de muestra
with Session() as session:
    stmt = (
        select(SampleType.type_name, func.count(Sample.id).label("total"))
        .join(Sample, Sample.id_sample_type == SampleType.id)
        .group_by(SampleType.id, SampleType.type_name)
        .order_by(desc("total"))
    )

    rows = session.execute(stmt).all()

    for type_name, total in rows:
        print(f"{type_name:<25} {total}")

In [ ]:
# Temperatura media registrada por cada muestra
with Session() as session:
    stmt = (
        select(
            Sample.code,
            func.avg(LogTemperature.temperature).label("temp_media"),
            func.count(LogTemperature.id).label("num_registros")
        )
        .join(LogTemperature, LogTemperature.id_sample == Sample.id)
        .group_by(Sample.id, Sample.code)
        .order_by(Sample.code)
    )

    rows = session.execute(stmt).all()

    for code, temp_media, num in rows:
        print(f"{code} | temp media: {float(temp_media):.2f}°C | {num} registro(s)")

In [ ]:
# having → pacientes con MÁS DE 1 muestra
with Session() as session:
    stmt = (
        select(Patient.name, Patient.lastname, func.count(Sample.id).label("total"))
        .join(Sample, Sample.id_patient == Patient.id)
        .group_by(Patient.id, Patient.name, Patient.lastname)
        .having(func.count(Sample.id) > 1)
        .order_by(desc("total"))
    )

    rows = session.execute(stmt).all()

    for name, lastname, total in rows:
        print(f"{name} {lastname} → {total} muestras")

---
## 1️⃣2️⃣ Paginación — `limit` + `offset`
`offset` = cuántos registros saltar. `limit` = cuántos traer.

In [ ]:
page_size = 3
page = 1  # Primera página

with Session() as session:
    stmt = (
        select(Sample)
        .order_by(Sample.code)
        .limit(page_size)
        .offset((page - 1) * page_size)
    )

    samples = session.scalars(stmt).all()

    print(f"--- Página {page} ---")
    for s in samples:
        print(s.code, s.status)

In [ ]:
page_size = 3
page = 2  # Segunda página

with Session() as session:
    stmt = (
        select(Sample)
        .order_by(Sample.code)
        .limit(page_size)
        .offset((page - 1) * page_size)
    )

    samples = session.scalars(stmt).all()

    print(f"--- Página {page} ---")
    for s in samples:
        print(s.code, s.status)

---
## 1️⃣3️⃣ `scalars()` vs `execute()` vs `mappings()`
| Método | Devuelve | Cuándo usarlo |
|---|---|---|
| `scalars(stmt).all()` | lista de objetos ORM | `select(Modelo)` completo |
| `execute(stmt).all()` | lista de tuplas (`Row`) | columnas sueltas, JOINs, agregados |
| `execute(stmt).mappings().all()` | lista de dicts | columnas sueltas, acceso por nombre |

In [ ]:
# scalars() → objetos ORM, accedes por atributo
with Session() as session:
    stmt = select(Sample)
    samples = session.scalars(stmt).all()
    print(type(samples[0]))  # <class 'domain.models.Sample'>
    print(samples[0].code)   # acceso por atributo

In [ ]:
# execute() → tuplas Row, accedes por posición o nombre de columna
with Session() as session:
    stmt = select(Sample.code, Sample.status, Sample.volume)
    rows = session.execute(stmt).all()

    for row in rows:
        print(row.code, row.status, float(row.volume))

In [ ]:
# mappings() → dicts, accedes por clave string
# Útil cuando necesitas serializar o pasar datos a otra función
with Session() as session:
    stmt = (
        select(
            Patient.name.label("paciente"),
            func.count(Sample.id).label("muestras")
        )
        .outerjoin(Sample, Sample.id_patient == Patient.id)
        .group_by(Patient.id, Patient.name)
        .order_by(Patient.name)
    )

    result = session.execute(stmt).mappings().all()

    for row in result:
        print(row["paciente"], row["muestras"])

---
## 1️⃣4️⃣ Modificar y eliminar registros
El flujo siempre es: **buscar → modificar objeto → `commit()`**

SQLAlchemy rastrea automáticamente los cambios en los objetos cargados en sesión (*unit of work*). Al hacer `commit()`, genera el SQL necesario y lo envía a la DB.

In [ ]:
# UPDATE: cambiar el estado de una muestra
# 1. Buscamos el objeto
# 2. Modificamos el atributo directamente
# 3. commit() → SQLAlchemy genera UPDATE sample SET status='in_process' WHERE id=...
with Session() as session:
    sample = session.scalar(select(Sample).where(Sample.code == "M001"))

    if sample:
        print(f"Estado antes: {sample.status}")
        sample.status = "in_process"  # Solo modificamos el objeto Python
        session.commit()              # Aquí se actualiza la DB
        print(f"Estado después: {sample.status}")

In [ ]:
# UPDATE: desactivar un paciente
with Session() as session:
    patient = session.scalar(select(Patient).where(Patient.code == "P005"))

    if patient:
        patient.active = False
        session.commit()
        print(f"{patient.name} → active: {patient.active}")

In [ ]:
# INSERT: añadir un registro de temperatura a una muestra existente
from datetime import datetime

with Session() as session:
    sample = session.scalar(select(Sample).where(Sample.code == "M001"))

    if sample:
        nuevo_log = LogTemperature(
            temperature=-20.5,
            reading_date=datetime.now(),
            id_sample=sample.id
        )
        session.add(nuevo_log)
        session.commit()
        print(f"Log añadido: {nuevo_log.id} | {nuevo_log.temperature}°C")

In [ ]:
# DELETE: eliminar el último log de temperatura insertado
with Session() as session:
    stmt = (
        select(LogTemperature)
        .order_by(desc(LogTemperature.id))
        .limit(1)
    )
    log = session.scalar(stmt)

    if log:
        print(f"Eliminando log {log.id} | {log.temperature}°C")
        session.delete(log)
        session.commit()
        print("Eliminado")

In [ ]:
# Añadir un protocolo a una muestra (relación N:M)
# SQLAlchemy gestiona la tabla intermedia sample_protocol automáticamente
with Session() as session:
    sample = session.scalar(
        select(Sample)
        .options(selectinload(Sample.protocols))
        .where(Sample.code == "M001")
    )
    protocol = session.scalar(select(Protocol).where(Protocol.code == "PROT-001"))

    if sample and protocol:
        if protocol not in sample.protocols:
            sample.protocols.append(protocol)  # SQLAlchemy inserta en sample_protocol
            session.commit()
            print(f"Protocolo {protocol.code} añadido a {sample.code}")
        else:
            print("Ya estaba asignado")